# Project 2: Text Similarity & Duplicate Detection (G05)

## Notebook 3 (Approach 3): Fine-Tuning BERT (Cross-Encoder)

Step 1: Setup HuggingFace Environment

Installation:

In [ ]:
!pip install transformers datasets evaluate accelerate

Libraries:

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
import os

Step 2: Data Preparation & Tokenization

In [9]:
import os
import pandas as pd

print("1. Loading dataset from local CSV file...")

data_path = "questions.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Could not find {data_path!r} in the current folder. "
        "Put your dataset CSV next to this notebook or change `data_path`."
    )

df = pd.read_csv(data_path)

# Keep only the columns we actually need for the model
required_cols = ["question1", "question2", "is_duplicate"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}. Found: {list(df.columns)[:20]}...")

# Drop nulls to prevent the tokenizer error
df = df[required_cols].dropna()

# SAMPLE THE DATA so your computer survives the BERT fine-tuning
df = df.sample(n=10000, random_state=42) 

print(f"-> Dataset loaded successfully with {df.shape[0]} rows!\n")

1. Loading dataset from local CSV file...
-> Dataset loaded successfully with 10000 rows!



In [ ]:
# Force columns to string to keep the tokenizer happy
df['question1'] = df['question1'].astype(str)
df['question2'] = df['question2'].astype(str)

# Load tokenizer (using a lightweight BERT for faster training)
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Convert pandas dataframe to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df[['question1', 'question2', 'is_duplicate']])
hf_dataset = hf_dataset.rename_column("is_duplicate", "label")

# Split dataset
hf_dataset = hf_dataset.train_test_split(test_size=0.2)

# Tokenization function for pairs of text
def tokenize_function(example):
    return tokenizer(example["question1"], example["question2"], truncation=True, padding="max_length", max_length=128)

# Apply tokenization
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step 3: Model Setup and Metrics

In [11]:
# Load pre-trained model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Define evaluation metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step 4: Training via Trainer API

In [ ]:
# Load pre-trained model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Define evaluation metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,  # <-- CHANGED THIS LINE
    compute_metrics=compute_metrics,
)

print("Starting training... This may take a while!")
# Start fine-tuning
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training... This may take a while!


c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Step 5: Final Evaluation

In [ ]:
# Evaluate on the test set
results = trainer.evaluate()
print("Final Evaluation Results:", results)